# metabeta — priors and prior sensitivity

This notebook demonstrates how to control and inspect priors in **metabeta**:

1. **Simulate** a Bernoulli dataset from a known prior — so we know the ground truth.
2. **Specify explicit priors** and run a batched prior-sensitivity analysis in a single `sample()` call.
3. **Plot posterior parameters** with prior overlays for each prior variant.
4. **Compare log-probabilities** assigned to posterior samples under each prior variant.

> **Setup (run once):** The cell below assembles a joint checkpoint from local training runs.
> In a future release, pre-trained weights will be downloaded automatically on first use.

In [ ]:
# Run this cell once to build the joint checkpoint
from pathlib import Path
from metabeta.utils.api import joinCheckpoints
from metabeta.utils.experiments import CHECKPOINT_DIR

OUTPUT_PATH = Path('/tmp/joint_bernoulli_v1.pt')

if not OUTPUT_PATH.exists():
    ckpt_dir = CHECKPOINT_DIR
    checkpoints = [
        ckpt_dir / 'data=small-b-mixed_model=large_seed=2',
        ckpt_dir / 'data=small-b-mixed_model=large_seed=3',
        ckpt_dir / 'data=small-b-mixed_model=large_seed=4',
        ckpt_dir / 'data=small-b-mixed_model=large_seed=5',
    ]
    joinCheckpoints(
        checkpoints,
        output_path=OUTPUT_PATH,
        ids=['b_s2', 'b_s3', 'b_s4', 'b_s5'],
    )
    print(f'Joint checkpoint saved to {OUTPUT_PATH}')
else:
    print(f'Joint checkpoint already exists at {OUTPUT_PATH}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from metabeta.models.api import Api

mb = Api('/tmp/joint_bernoulli_v1.pt', device='cpu')

## 1. Simulate a Bernoulli dataset

Rather than loading a fixed dataset, we generate data from a known prior — so we can check
how well the posterior recovers the true parameters and how prior choice affects inference.

We use the full simulation pipeline:
- `hypersample` draws hyperparameters (scale and family of each prior component)
- `Prior` uses those hyperparameters to sample true parameters
- `Scammer` generates covariates via a structural causal model (realistic non-linear dependencies)
- `Simulator` produces binary outcomes under the Bernoulli likelihood

The model has **4 fixed effects** (intercept + 3 predictors) and **2 correlated random effects**
(random intercept + random slope on X1).

In [ ]:
from metabeta.simulation import Prior, Scammer
from metabeta.simulation.prior import hypersample
from metabeta.simulation.simulator import Simulator

SEED = 7
D = 4      # fixed effects: intercept + X1 + X2 + X3
Q = 2      # random effects: intercept + X1 slope
M = 30     # number of groups
FORMULA = 'y ~ X1 + X2 + X3 + (1 + X1 | group)'
LIKELIHOOD_FAMILY = 1  # Bernoulli

rng = np.random.default_rng(SEED)

# ensure correlated random effects for a richer example
hparams = hypersample(rng, D, Q, likelihood_family=LIKELIHOOD_FAMILY)
while float(hparams['eta_rfx']) == 0.0:
    rng = np.random.default_rng(rng.integers(0, 2**31))
    hparams = hypersample(rng, D, Q, likelihood_family=LIKELIHOOD_FAMILY)

prior = Prior(rng, hparams)
scammer = Scammer(rng)
ns = rng.integers(8, 20, size=M)
sim = Simulator(rng, prior, scammer, ns)
ds = sim.sample()

# build DataFrame (drop intercept column X[:, 0] == 1)
df = pd.DataFrame(ds['X'][:, 1:], columns=['X1', 'X2', 'X3'])
df['y'] = ds['y'].astype(int)
df['group'] = ds['groups']

print(f'Dataset: N={len(df)}, M={df["group"].nunique()} groups')
print(f'Outcome prevalence: {df["y"].mean():.2%}')
print(f'LKJ eta_rfx: {float(hparams["eta_rfx"]):.2f}')
print()
print('True parameters:')
print(f'  ffx  (intercept, X1, X2, X3):  {np.round(ds["ffx"], 3)}')
print(f'  sigma_rfx:                       {np.round(ds["sigma_rfx"], 3)}')
print(f'  corr(intercept, X1 slope):       {ds["corr_rfx"][1, 0]:.3f}')

In [ ]:
from metabeta.plotting import plotDataset

g = plotDataset(df[['X1', 'X2', 'X3']], title='Simulated predictors (SCM)')
plt.show()

## 2. Prior sensitivity analysis

We compare three prior specifications in a **single `sample()` call** by passing a named
dictionary.  metabeta stacks them into a batch, so all three are processed together:

| Variant | Description |
|---|---|
| `default` | Flat prior — model uses its learned default |
| `informative` | Moderately tight priors consistent with logit-scale effects |
| `skeptical` | Very tight prior that shrinks all effects toward zero |

Priors use the **term-based schema**:
- `fixed` maps each formula term to a `{"tau": value}` scale spec
- `random_sd` maps each random-effect SD term to a `{"tau": value}` scale spec
- `corr_rfx` takes `{"eta": value}` for the LKJ concentration

> **Note:** use `{}` (not `None`) for the flat prior variant — `None` is not recognized as a
> named-collection entry and would collapse the batch to a single prior.

In [ ]:
PRIORS = {
    'default': {},
    'informative': {
        'fixed': {
            'Intercept': {'tau': 1.0},
            'X1':        {'tau': 0.8},
            'X2':        {'tau': 0.8},
            'X3':        {'tau': 0.8},
        },
        'random_sd': {
            'Intercept': {'tau': 0.6},
            'X1':        {'tau': 0.6},
        },
        'corr_rfx': {'eta': 1.5},
    },
    'skeptical': {
        'fixed': {
            'Intercept': {'tau': 0.5},
            'X1':        {'tau': 0.3},
            'X2':        {'tau': 0.3},
            'X3':        {'tau': 0.3},
        },
        'random_sd': {
            'Intercept': {'tau': 0.3},
            'X1':        {'tau': 0.3},
        },
        'corr_rfx': {'eta': 2.0},
    },
}

result = mb.sample(
    df,
    formula=FORMULA,
    priors=PRIORS,
    n_samples=1000,
    diagnostics=True,
)

print(f'Proposal batch size: {result.proposal.samples_g.shape[0]}')

## 3. Posterior summaries

In [ ]:
for i, name in enumerate(PRIORS):
    print(f'\n{"="*60}')
    print(f'Prior: {name}')
    print('='*60)
    print(mb.posteriorSummary(result, batch_index=i))

## 4. Posterior parameter plots with prior overlays

The pair grids show:
- **Green**: marginal posterior KDE / 2D contours
- **Blue**: analytical prior PDF
- **Purple dashed**: ground-truth parameter values

The bottom-right column shows the correlation parameter ρ between the random intercept and
random X1 slope — included automatically when `q ≥ 2` and the model used a LKJ prior.

In [ ]:
# ground-truth vector: [ffx..., sigma_rfx..., rho]
truth = np.concatenate([ds['ffx'], ds['sigma_rfx'], [ds['corr_rfx'][1, 0]]])

for i, name in enumerate(PRIORS):
    g = mb.plotParameters(result, index=i, with_prior=True, truth=truth)
    g.figure.suptitle(f'Posterior  ·  prior: {name}', fontsize=18, y=1.02)
    plt.show()

## 5. Log-probability of posterior samples

`mb.log_prob()` scores any parameter vector against the conditioned posterior — useful for
model criticism, importance weighting, or comparing how well a candidate point sits under
the posterior.

Here we look at the **log-probabilities already assigned to posterior samples** during
`sample()`.  `result.proposal.log_prob_g[i]` gives the S log-density values under prior
variant `i`'s conditioned flow.

Tighter priors produce more concentrated posteriors → samples land in denser regions → higher
average log-prob.  Flat priors spread the mass → lower average log-prob.

In [ ]:
print('Log-prob of posterior samples (global flow):')
print(f'{"Prior":>14}  {"mean":>8}  {"median":>8}  {"std":>8}')
print('-' * 44)
lp_data = []
for i, name in enumerate(PRIORS):
    lp = result.proposal.log_prob_g[i].float()  # (S,)
    lp_data.append(lp.numpy())
    print(f'{name:>14}  {lp.mean().item():8.2f}  {lp.median().item():8.2f}  {lp.std().item():8.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3), dpi=150)
ax.boxplot(lp_data, labels=list(PRIORS.keys()), patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.5),
           medianprops=dict(color='darkblue', lw=2))
ax.set_ylabel('Global log-prob')
ax.set_title('Distribution of posterior sample log-probs by prior variant')
ax.grid(axis='y', alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

### Scoring an external parameter vector with `mb.log_prob()`

To score a specific parameter vector (e.g., from a different model or the ground truth),
prepare a batch with `ffx`, `sigma_rfx`, and `rfx` keys and call `mb.log_prob(batch)`.
The batch must be at the collated stage — use `mb.prepareData(...)` and then inject the
parameter tensors before passing to `mb.log_prob()`.

Below we score the **posterior mean** from each prior variant under the **default prior**
conditioning.

In [ ]:
# Prepare a base single-prior batch (priors=None → model default)
base_batch = mb.prepareData(df, formula=FORMULA)
print('Base batch X shape:', base_batch['X'].shape)  # (B=1, M_max, N_max, d)

# For each prior variant, score its posterior mean under the default conditioning
print('\nLog-prob of posterior mean under default conditioning:')
for i, name in enumerate(PRIORS):
    prop = result.proposal

    # Posterior mean parameters (B=1)
    ffx_pm = prop.ffx[i].mean(0, keepdim=True)           # (1, d)
    srfx_pm = prop.sigma_rfx[i].mean(0, keepdim=True)    # (1, q)
    rfx_pm = prop.rfx[i].mean(1)                          # (m, q)

    M_max = base_batch['X'].shape[1]
    q = rfx_pm.shape[-1]
    rfx_padded = torch.zeros(1, M_max, q)                 # (1, M_max, q)
    rfx_padded[0, :rfx_pm.shape[0], :] = rfx_pm

    scored_batch = {
        k: v.clone() if torch.is_tensor(v) else v
        for k, v in base_batch.items()
    }
    scored_batch['ffx'] = ffx_pm
    scored_batch['sigma_rfx'] = srfx_pm
    scored_batch['rfx'] = rfx_padded

    lp_result = mb.log_prob(scored_batch)
    lp_global = lp_result.log_probs['global'].item()
    print(f'  {name:>12s}: log_prob(global) = {lp_global:.2f}')